<a href="https://colab.research.google.com/github/LukeXboy/Healthcare-Readmission-Data-Science-Project/blob/main/Project%20Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Healthcare Readmission Prediction

**Question:** Can we predict a recorded 30-day readmission for an inpatient encounter involving diabetes using data available by discharge?

**Source:** [UCI Diabetes 130-US Hospitals, 1999–2008](https://archive.ics.uci.edu/dataset/296/diabetes%2B130-us%2Bhospitals%2Bfor%2Byears%2B1999-2008), DOI 10.24432/C5230J (CC BY 4.0). Retrospective historical US data; not a deployable clinical system.

**Today:** Explore the raw data without modifying it, and report observations. Don't fit a model or drop missing features yet.


In [1]:
from pathlib import Path
import pandas as pd


df = pd.read_excel('/content/diabetic_data.csv.xlsx', na_values=['?'])
print('Loaded dataset:', '/content/diabetic_data.csv.xlsx')
print('Shape:', df.shape)
display(df.head())


Loaded dataset: /content/diabetic_data.csv.xlsx
Shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Part 1 — Data grain and missing values


In [2]:
print('Number of rows and columns:', df.shape)
print('Column names:', df.columns)
print('Number of distinct encounter_id values:', df['encounter_id'].nunique())
print('Number of distinct patient_nbr values:', df['patient_nbr'].nunique())
print('10 columns with the highest percentage of missing values:', df.isna().mean().sort_values(ascending=False).head(10))
print('Raw readmitted class counts and percentages:', df['readmitted'].value_counts(normalize=True))

Number of rows and columns: (101766, 50)
Column names: Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')
Number of dist

##Part 2 - Train/Test Split





In [3]:
from sklearn.model_selection import GroupShuffleSplit
#using groupshufflesplit to prevent same patients appearing in both sets
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        groups=df['patient_nbr']
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

In [4]:
train_patients = set(train_df['patient_nbr'])
test_patients = set(test_df['patient_nbr'])

overlap = train_patients.intersection(test_patients)

print("Training encounters:", len(train_df))
print("Test encounters:", len(test_df))
print("Training patients:", len(train_patients))
print("Test patients:", len(test_patients))
print("Overlapping patients:", len(overlap))

Training encounters: 81613
Test encounters: 20153
Training patients: 57214
Test patients: 14304
Overlapping patients: 0


In [5]:
#check class distribution in train and test set
print("Training distribution:")
print(
    train_df['readmitted'].value_counts(
        normalize=True
    )
)

print("\nTest distribution:")
print(
    test_df['readmitted'].value_counts(
        normalize=True
    )
)

Training distribution:
readmitted
NO     0.537292
>30    0.349907
<30    0.112801
Name: proportion, dtype: float64

Test distribution:
readmitted
NO     0.546519
>30    0.346747
<30    0.106733
Name: proportion, dtype: float64


In [6]:
#make binary target variable: readmitted_30d
train_df['readmitted_30d'] = (train_df['readmitted'] == '<30').astype(int)
test_df['readmitted_30d'] = (test_df['readmitted'] == '<30').astype(int)
#check percentage of positive and negative class in each set
print("Training distribution:")
print(
    train_df['readmitted_30d'].value_counts(
        normalize=True
    )
)

print("\nTest distribution:")
print(
    test_df['readmitted_30d'].value_counts(
        normalize=True
    )
)

Training distribution:
readmitted_30d
0    0.887199
1    0.112801
Name: proportion, dtype: float64

Test distribution:
readmitted_30d
0    0.893267
1    0.106733
Name: proportion, dtype: float64


In both train and test dataset, the class distribution is very imbalanced (about 11% are postive class). Therefore, later when we evaluate our model performance, we need to use PR-AUC, ROC-AUC, precision, recall, and F1 rather than relying on accuracy alone.

##Part 3 - Data Preprocessing

In [7]:
import numpy as np
import pandas as pd

#seperate numerical and categorical features
numeric_features = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses'
]

categorical_features = [
    'age',
    'gender',
    'admission_type_id',
    'admission_source_id',
    'change',
    'diabetesMed'
]

feature_columns = numeric_features + categorical_features

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df['readmitted_30d'].copy()
y_test = test_df['readmitted_30d'].copy()

# Standardize missing-value representations
X_train = X_train.replace('?', np.nan)
X_test = X_test.replace('?', np.nan)

In [8]:
#impute missing values for both numerical and categorical features
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()) #standardize numerical data
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore'
    )) #one-hot code to convert categorical data into numerical
])

#build final preprocessor
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])


In [9]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nNumeric feature types:")
print(X_train[numeric_features].dtypes)

print("\nCategorical feature types:")
print(X_train[categorical_features].dtypes)

print("\nMissing values in training:")
print(X_train.isna().sum().sort_values(ascending=False))

X_train shape: (81613, 14)
X_test shape: (20153, 14)

Numeric feature types:
time_in_hospital      int64
num_lab_procedures    int64
num_procedures        int64
num_medications       int64
number_outpatient     int64
number_emergency      int64
number_inpatient      int64
number_diagnoses      int64
dtype: object

Categorical feature types:
age                    object
gender                 object
admission_type_id       int64
admission_source_id     int64
change                 object
diabetesMed            object
dtype: object

Missing values in training:
time_in_hospital       0
num_lab_procedures     0
num_procedures         0
num_medications        0
number_outpatient      0
number_emergency       0
number_inpatient       0
number_diagnoses       0
age                    0
gender                 0
admission_type_id      0
admission_source_id    0
change                 0
diabetesMed            0
dtype: int64


In [10]:
from sklearn.model_selection import GroupShuffleSplit
#create validation set
val_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=123
)

dev_train_idx, val_idx = next(
    val_splitter.split(
        train_df,
        groups=train_df['patient_nbr']
    )
)

dev_train_df = train_df.iloc[dev_train_idx].copy()
val_df = train_df.iloc[val_idx].copy()

In [11]:
train_val_overlap = set(dev_train_df['patient_nbr']).intersection(set(val_df['patient_nbr']))
train_test_overlap = set(dev_train_df['patient_nbr']).intersection(set(test_df['patient_nbr']))
val_test_overlap = set(val_df['patient_nbr']).intersection(set(test_df['patient_nbr']))

print('The encounter counts for each set are:', dev_train_df['encounter_id'].nunique(), val_df['encounter_id'].nunique(), test_df['encounter_id'].nunique())
print('The patient counts for each set are:', dev_train_df['patient_nbr'].nunique(), val_df['patient_nbr'].nunique(), test_df['patient_nbr'].nunique())
print('The patient overlap for each set are:', len(train_val_overlap), len(train_test_overlap), len(val_test_overlap))
print('The positive class percentage for each set are:', dev_train_df['readmitted_30d'].mean()*100, val_df['readmitted_30d'].mean()*100, test_df['readmitted_30d'].mean()*100)

The encounter counts for each set are: 65274 16339 20153
The patient counts for each set are: 45771 11443 14304
The patient overlap for each set are: 0 0 0
The positive class percentage for each set are: 11.286270184146826 11.25527878083114 10.673348881059892


##Part 4 - Logistic Regression model

In [12]:
import numpy as np

X_dev = dev_train_df[feature_columns].copy()
X_val = val_df[feature_columns].copy()

y_dev = dev_train_df['readmitted_30d'].copy()
y_val = val_df['readmitted_30d'].copy()

X_dev = X_dev.replace('?', np.nan)
X_val = X_val.replace('?', np.nan)

print(X_dev.shape, X_val.shape)
print(y_dev.mean(), y_val.mean())

(65274, 14) (16339, 14)
0.11286270184146827 0.1125527878083114


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

In [14]:
from sklearn.metrics import roc_auc_score, average_precision_score

# 1. Fit the entire pipeline on X_dev and y_dev
lr_model = lr_pipeline.fit(X_dev, y_dev)

# 2. Predict the probability of class 1 for X_val
lr_preds = lr_model.predict_proba(X_val)[:, 1]

# 3. Calculate validation ROC-AUC
val_roc_auc = roc_auc_score(y_val, lr_preds)

# 4. Calculate validation Average Precision (AP)
val_ap = average_precision_score(y_val, lr_preds)

# 5. Print both scores
print("Validation ROC-AUC:", val_roc_auc)
print("Validation Average Precision (AP):", val_ap)


Validation ROC-AUC: 0.6339023832292663
Validation Average Precision (AP): 0.20002209090301487


In [16]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Convert probabilities into binary labels
threshold = 0.5
y_pred = (lr_preds >= threshold).astype(int)
val_pr = precision_score(y_val, y_pred)
val_re = recall_score(y_val, y_pred)
val_f1 = f1_score(y_val, y_pred)
confusion_matrix = confusion_matrix(y_val, y_pred)

print("Confusion Matrix:")
print(confusion_matrix)
print("Validation Precision:", val_pr)
print("Validation Recall:", val_re)
print("Validation F1 Score:", val_f1)

Confusion Matrix:
[[14470    30]
 [ 1808    31]]
Validation Precision: 0.5081967213114754
Validation Recall: 0.016856987493202826
Validation F1 Score: 0.03263157894736842


From the results above, we can see that the model performance on validation set is very poor. Therefore, we need to adjust our threshold and choose the best value.

In [18]:
threshold_list = [0.1,0.2,0.3,0.4,0.5]
val_pr_list = []
val_re_list = []
val_f1_list = []
for tr in threshold_list:
  y_pred = (lr_preds >= tr).astype(int)
  val_pr_list.append(precision_score(y_val, y_pred))
  val_re_list.append(recall_score(y_val, y_pred))
  val_f1_list.append(f1_score(y_val, y_pred))

pd.DataFrame({'Threshold':threshold_list, 'Precision':val_pr_list, 'Recall':val_re_list, 'F1':val_f1_list})

,Threshold,Precision,Recall,F1
0,0.1,0.147948,0.654704,0.241355
1,0.2,0.275120,0.125068,0.171963
2,0.3,0.420849,0.059271,0.103908
3,0.4,0.459677,0.030995,0.058074
4,0.5,0.508197,0.016857,0.032632


I wouldn't choose threshold 0.10 solely because it has the highest F1-score. Its low precision means a large number of unnecessary follow-ups, which could overwhelm hospital resources. I would first establish the available intervention capacity and the costs of false positives and false negatives, then evaluate precision and recall at operationally feasible thresholds. I would also consider ranking patients by predicted risk and selecting the highest-risk group within our capacity.

##Part 5 - Random Forest model

In [19]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

# 1. Fit rf_pipeline using X_dev and y_dev
rf_model = rf_pipeline.fit(X_dev, y_dev)

# 2. Predict positive-class probabilities on X_val
rf_preds = rf_model.predict_proba(X_val)[:, 1]

# 3. Calculate validation ROC-AUC
val_roc_auc = roc_auc_score(y_val, rf_preds)

# 4. Calculate validation Average Precision
val_ap = average_precision_score(y_val, rf_preds)

# 5. Print both metrics
print("Validation ROC-AUC:", val_roc_auc)
print("Validation Average Precision (AP):", val_ap)

Validation ROC-AUC: 0.6416704355815567
Validation Average Precision (AP): 0.19968077787671612


From the results, we can see that there is a slight improvement on ROC-AUC score of random forest model compared to logistic regression model. However, this modest change does not show that a more complex model is better.

##Part 6 - XGBoost Classification

In [20]:
from xgboost import XGBClassifier


xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

# 1. Fit the pipeline on X_dev and y_dev
xgb_model = xgb_pipeline.fit(X_dev, y_dev)

# 2. Predict probabilities for X_val
xbg_preds = xgb_model.predict_proba(X_val)[:, 1]

# 3. Calculate ROC-AUC
xbg_roc_auc = roc_auc_score(y_val, xbg_preds)

# 4. Calculate Average Precision
xbg_ap = average_precision_score(y_val, xbg_preds)

# 5. Print both metrics
print("Validation ROC-AUC:", xbg_roc_auc)
print("Validation Average Precision (AP):", xbg_ap)


Validation ROC-AUC: 0.6433619283343647
Validation Average Precision (AP): 0.20434643689389265


In [21]:
#check overfitting

# 1. Predict probabilities on X_dev
xgb_train_probs = xgb_model.predict_proba(X_dev)[:, 1]

# 2. Predict probabilities on X_val
xbg_val_probs = xgb_model.predict_proba(X_val)[:, 1]

# 3. Calculate training ROC-AUC and AP
xgb_train_roc_auc = roc_auc_score(y_dev, xgb_train_probs)

# 4. Calculate validation ROC-AUC and AP
xgb_train_ap = average_precision_score(y_dev, xgb_train_probs)


# 5. Display results in a DataFrame
pd.DataFrame({
    'Dataset': ['Training', 'Validation'],
    'ROC-AUC': [xgb_train_roc_auc, xbg_roc_auc],
    'Average Precision (AP)': [xgb_train_ap, xbg_ap]
})


,Dataset,ROC-AUC,Average Precision (AP)
0,Training,0.661542,0.224827
1,Validation,0.643362,0.204346


##Part 6.2 - XGBoost Hyperparameter Tuning with Cross Validation

In [22]:
from sklearn.model_selection import (
    StratifiedGroupKFold,
    RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

#since each patient might have multiple encounters, we use StratifiedGroupKFold to preserve the positive class percentage in each fold while keeping encounters from the same patients together
cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

tuning_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

param_distributions = {
    'classifier__max_depth': [2, 3, 4],
    'classifier__min_child_weight': [1, 5, 10],
    'classifier__subsample': [0.8, 1.0],
    'classifier__reg_lambda': [1, 5, 10]
}


In [23]:

search = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=param_distributions,
    n_iter=6,
    scoring='average_precision',
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=1
)

# 1. Fit the search on X_dev and y_dev.
# Important: pass patient_nbr as groups.
xgb_models = search.fit(X_dev, y_dev, groups=dev_train_df['patient_nbr'])

# 2. Print the best hyperparameters.
print('The best hyperparameters combination is:',xgb_models.best_params_)

# 3. Print the best mean cross-validation AP.
print('The best mean cross_validation AP is:', xgb_models.best_score_)

# 4. Predict probabilities on X_val
# using the best fitted pipeline.
best_xgb_model = xgb_models.best_estimator_
xgb_cv_preds = best_xgb_model.predict_proba(X_val)[:, 1]

# 5. Calculate validation ROC-AUC and AP.
xgb_cv_roc_auc = roc_auc_score(y_val, xgb_cv_preds)
xgb_cv_ap = average_precision_score(y_val, xgb_cv_preds)

# 6. Print both metrics.
print("Validation ROC-AUC:", xgb_cv_roc_auc)
print("Validation Average Precision (AP):", xgb_cv_ap)


Fitting 3 folds for each of 6 candidates, totalling 18 fits
The best hyperparameters combination is: {'classifier__subsample': 0.8, 'classifier__reg_lambda': 1, 'classifier__min_child_weight': 10, 'classifier__max_depth': 4}
The best mean cross_validation AP is: 0.19831129593007657
Validation ROC-AUC: 0.6424888901389436
Validation Average Precision (AP): 0.20192806953944917
